# Grad-CAM: Visualizing What Your Model "Sees"

**Grad-CAM** (Gradient-weighted Class Activation Mapping) is a technique that shows *which parts of an image* your neural network found important when making its prediction. Think of it like a "heat map": warmer colors (red) show the regions that influenced the model's decision most.

**In this notebook you will:**
1. Train a simple CNN on MNIST handwritten digits (your own model)
2. Apply Grad-CAM to see what your model focuses on
3. Apply Grad-CAM to a pre-trained ResNet on a real photo

*Requires: PyTorch, torchvision, matplotlib, numpy*

## 1. Setup and Imports

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from scipy.ndimage import zoom
import matplotlib.pyplot as plt
import numpy as np

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Train a Simple CNN on MNIST (Your Model)

We'll train a small convolutional neural network on the MNIST digit dataset. This gives us a model we can inspect with Grad-CAM.

In [7]:
# Simple CNN for MNIST - we'll hook into the last conv layer (conv2) for Grad-CAM
class SimpleMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Load MNIST data
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=0)

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data\MNIST\raw\train-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data\MNIST\raw\train-labels-idx1-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data\MNIST\raw\t10k-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%

Extracting ./data\MNIST\raw\t10k-labels-idx1-ubyte.gz to ./data\MNIST\raw



In [8]:
# Train the model (a few epochs is enough for a demo)
model_mnist = SimpleMNISTCNN().to(device)
optimizer = torch.optim.Adam(model_mnist.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

model_mnist.train()
for epoch in range(3):
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_mnist(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/3, Loss: {total_loss/len(train_loader):.4f}")

model_mnist.eval()
print("Training complete!")

Epoch 1/3, Loss: 0.1676
Epoch 2/3, Loss: 0.0492
Epoch 3/3, Loss: 0.0347
Training complete!


## 3. Grad-CAM Implementation

Grad-CAM uses two things from the network:
1. **Activations** from a convolutional layer (what the network "saw")
2. **Gradients** of the predicted class score with respect to those activations (how important each part was)

We combine them to create a heatmap. We'll use PyTorch "hooks" to capture these without modifying the model.

In [ ]:
def grad_cam(model, input_tensor, target_layer, target_class=None):
    """
    Compute Grad-CAM heatmap for a given model and input.
    
    Args:
        model: The CNN model
        input_tensor: Single image (1, C, H, W)
        target_layer: The conv layer to use (e.g. model.conv2)
        target_class: Which class to explain. If None, uses the predicted class.
    
    Returns:
        heatmap: 2D numpy array (H, W) with values 0-1
    """
    activations = []
    gradients = []

    def forward_hook(module, input, output):
        activations.append(output.detach())

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0].detach())

    # Register hooks
    handle_fwd = target_layer.register_forward_hook(forward_hook)
    handle_bwd = target_layer.register_full_backward_hook(backward_hook)

    # Forward pass
    model.eval()
    output = model(input_tensor)
    if target_class is None:
        target_class = output.argmax(dim=1).item()

    # Zero gradients, then backward pass for the target class
    model.zero_grad()
    one_hot = torch.zeros_like(output)
    one_hot[0, target_class] = 1
    output.backward(gradient=one_hot, retain_graph=False)

    # Remove hooks
    handle_fwd.remove()
    handle_bwd.remove()

    # Get activations and gradients
    act = activations[0][0]  # (C, H, W)
    grad = gradients[0][0]   # (C, H, W)

    # Grad-CAM: global average of gradients as weights
    weights = grad.mean(dim=(1, 2))
    cam = (weights[:, None, None] * act).sum(dim=0)
    cam = torch.relu(cam)
    cam = cam.cpu().numpy()
    # Normalize to 0-1
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam, target_class

In [ ]:
def show_gradcam(image, heatmap, title="Grad-CAM"):
    """Overlay heatmap on image and display."""
    import matplotlib.pyplot as plt
    from scipy.ndimage import zoom

    # Resize heatmap to match image size if needed
    if heatmap.shape != image.shape[-2:]:
        zoom_factors = (image.shape[-2] / heatmap.shape[0], image.shape[-1] / heatmap.shape[1])
        heatmap = zoom(heatmap, zoom_factors, order=1)

    # Create overlay (heatmap in red)
    if len(image.shape) == 2:
        img = image
        overlay = np.stack([img, img, img], axis=-1)
    else:
        img = image.transpose(1, 2, 0) if image.shape[0] in (1, 3) else image
        overlay = img.copy() if img.ndim == 3 else np.stack([img, img, img], axis=-1)

    # Normalize overlay to 0-1 for display
    if overlay.max() > 1:
        overlay = overlay / overlay.max()
    heatmap_resized = np.clip(heatmap, 0, 1)

    # Apply colormap (red = high attention)
    plt.imshow(overlay)
    plt.imshow(heatmap_resized, cmap='jet', alpha=0.5)
    plt.title(title)
    plt.axis('off')
    plt.colorbar(plt.cm.ScalarMappable(cmap='jet'), shrink=0.6)
    plt.show()

## 4. Grad-CAM on Your Trained MNIST Model

Let's pick a few test images and see what regions your model focused on when making its predictions.

In [ ]:
# Get a few test samples
test_loader = DataLoader(test_data, batch_size=1, shuffle=True)
samples = [next(iter(test_loader)) for _ in range(4)]

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, (img, label) in enumerate(samples):
    img = img.to(device)
    heatmap, pred_class = grad_cam(model_mnist, img, model_mnist.conv2)
    img_np = img[0, 0].cpu().numpy()  # (28, 28)
    # Resize heatmap (7x7) to image size (28x28)
    heatmap = zoom(heatmap, (28/heatmap.shape[0], 28/heatmap.shape[1]), order=1)

    axes[0, i].imshow(img_np, cmap='gray')
    axes[0, i].set_title(f"True: {label.item()}, Pred: {pred_class}")
    axes[0, i].axis('off')

    axes[1, i].imshow(img_np, cmap='gray')
    axes[1, i].imshow(heatmap, cmap='jet', alpha=0.5)
    axes[1, i].set_title("Grad-CAM")
    axes[1, i].axis('off')

plt.suptitle("Your MNIST Model: What it looks at", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Grad-CAM on a Pre-trained ResNet

Now let's use a **pre-trained ResNet18** (trained on ImageNet with millions of images). We don't need to train it—we'll load it and run Grad-CAM on a sample image. This shows how Grad-CAM works on "real world" image recognition.

In [ ]:
# Load pre-trained ResNet18 (trained on ImageNet - 1000 classes)
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet = resnet.to(device)
resnet.eval()

# Get a sample image - we'll use CIFAR-10 (has cats, dogs, etc.) and resize to 224x224
# ResNet expects 224x224 RGB. You can also load your own image with: img = PIL.Image.open("your_image.jpg")
from PIL import Image
cifar = datasets.CIFAR10(root="./data", train=False, download=True)
img_pil = cifar[0][0]  # First test image
img_pil = img_pil.resize((224, 224), Image.BILINEAR)

In [ ]:
# Preprocess for ResNet (ImageNet normalization)
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
img_tensor = preprocess(img_pil).unsqueeze(0).to(device)

# Run Grad-CAM on ResNet. Target layer: last conv block (layer4)
target_layer = resnet.layer4[-1]
heatmap, pred_class = grad_cam(resnet, img_tensor, target_layer)

# ImageNet class index to name (simplified - full list has 1000 classes)
# pred_class is 0-999. We'll show the top prediction.
with torch.no_grad():
    logits = resnet(img_tensor)
    probs = torch.softmax(logits, dim=1)
    top5 = probs[0].topk(5)
print(f"Top prediction: class {pred_class} (confidence: {top5.values[0].item():.2%})")

In [ ]:
# Resize heatmap to 224x224 and overlay on image
img_display = np.array(img_pil) / 255.0  # For display (0-1)
heatmap_resized = zoom(heatmap, (224/heatmap.shape[0], 224/heatmap.shape[1]), order=1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_display)
axes[0].set_title("Original image")
axes[0].axis("off")

axes[1].imshow(heatmap_resized, cmap='jet')
axes[1].set_title("Grad-CAM heatmap")
axes[1].axis("off")

axes[2].imshow(img_display)
axes[2].imshow(heatmap_resized, cmap='jet', alpha=0.5)
axes[2].set_title("Overlay: What ResNet focused on")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## Summary

You've seen Grad-CAM in action on:
1. **Your trained MNIST model** – heatmaps show which strokes the network used to recognize digits
2. **A pre-trained ResNet** – heatmaps show which regions (e.g., animal faces, objects) influenced the prediction

**Try it yourself:** Replace `img_pil = cifar[0][0]` with your own image:  
`img_pil = Image.open("path/to/your/image.jpg")` and re-run the ResNet cells.